# E003 — Hyperparameter search for DCN-v2 and LightGBM

**Hypotheses:** none (supporting experiment) ([docs/hypotheses.md](../../docs/hypotheses.md))

**Question:** Which configurations to carry into the model comparison, chosen without touching the validation day?

**Method:** Optuna TPE, 60 DCN-v2 and 40 LightGBM trials, trained on 10-21..10-26 and selected on 10-27 NE. The validation day (10-28) stays clean for early stopping, calibration and comparisons.

In [1]:
from charade.analysis.experiment import setup

ctx = setup()
f"mode: {'smoke (fixture, tiny model)' if ctx.smoke else 'full data'}"

'mode: full data'

The full search takes ~16 minutes on the GTX 1660 SUPER and GPU training is not bit-reproducible, so in
full mode this notebook shows the committed search results; set `CHARADE_RERUN_TUNING=1` to rerun it.

In [2]:
import os

import polars as pl

from charade.models import tune

out = ctx.reports / "tuning"
if ctx.smoke:
    tune.run(trials_dcn=2, trials_gbdt=2, data_dir=ctx.data_dir, out=out)
elif os.environ.get("CHARADE_RERUN_TUNING") == "1":
    tune.run(out=out)
dcn, gbdt = pl.read_csv(out / "dcn.csv"), pl.read_csv(out / "gbdt.csv")
dcn.head(5)

/home/lucas/Developer/personal/take_home_assigment/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


trial,value,hidden,embedding_dim,cross_layers,cross_rank,dropout,lr,weight_decay,batch_size,character_id_dropout,best_step
i64,f64,str,i64,i64,i64,f64,f64,f64,i64,f64,i64
34,0.874582,"""256-128""",24,2,64,0.299255,0.002934,0.000013,2048,0.081288,312
39,0.874804,"""512-256""",24,3,32,0.181916,0.00238,3.5316e-7,2048,0.013046,312
55,0.874813,"""512-256""",24,2,32,0.191617,0.002444,0.000006,2048,0.020202,312
49,0.874841,"""512-256""",24,3,32,0.200711,0.002519,0.000004,2048,0.097183,312
58,0.874862,"""512-256""",24,3,32,0.172891,0.002511,0.000002,2048,0.136569,312


In [3]:
gbdt.head(5)

trial,value,learning_rate,num_leaves,min_data_in_leaf,cat_smooth,cat_l2,feature_fraction,lambda_l2,best_iteration
i64,f64,f64,i64,i64,f64,f64,f64,f64,i64
39,0.878748,0.012745,415,63,47.466398,1.454938,0.41288,0.019472,471
27,0.878803,0.017306,433,52,50.215798,3.278558,0.478485,0.026514,324
31,0.878931,0.013092,387,78,42.883885,2.470043,0.455002,0.066793,458
38,0.879157,0.010981,510,88,77.294537,1.765012,0.490822,0.032142,462
35,0.879179,0.011413,337,72,62.214865,2.270006,0.415198,0.019964,505


In [4]:
shipped = ctx.settings.model
{"dcn": shipped.dcn.model_dump(), "gbdt": shipped.gbdt.model_dump()}

{'dcn': {'embedding_dim': 24,
  'cross_layers': 2,
  'cross_rank': 64,
  'hidden': [256, 128],
  'dropout': 0.3,
  'lr': 0.0029,
  'weight_decay': 1.3e-05,
  'batch_size': 2048,
  'max_epochs': 6,
  'evals_per_epoch': 4,
  'patience': 4,
  'character_id_dropout': 0.08,
  'seeds': [0, 1, 2]},
 'gbdt': {'learning_rate': 0.0127,
  'num_leaves': 415,
  'min_data_in_leaf': 63,
  'cat_smooth': 47.5,
  'cat_l2': 1.45,
  'feature_fraction': 0.41,
  'lambda_l2': 0.0195,
  'max_rounds': 3000}}

## Reading
- The top DCN trials agree on 24-dim embeddings, batch 2048 and lr ~2.5-2.9e-3; architecture choices
  beyond that move inner NE by < 0.0003.
- The best LightGBM trial is 0.004 NE behind the best DCN trial on the same inner split.

## Conclusion
Best inner-split NE: DCN-v2 0.8746 (24-dim embeddings, 2 cross layers, 256-128 MLP, dropout 0.3, lr 2.9e-3, batch 2048) vs LightGBM 0.8787; both configs ship in [tool.charade.model].

**Decision:** Copy the best trials into `[tool.charade.model]` (done by hand, citing the CSVs).